### tunning dos parâmetros, biblioteca do sklearn

conjunto dos parâmetros, com todas as combinações possíveis, com os melhores parâmetros.

#### O que é Parameter Tuning?

Parameter Tuning, ou ajuste de parâmetros, é um processo essencial no desenvolvimento de modelos de machine learning e algoritmos de otimização. Ele envolve a seleção dos melhores parâmetros para um determinado modelo, a fim de melhorar sua precisão e desempenho. Este processo é crucial para garantir que o modelo seja capaz de generalizar bem para novos dados e produzir resultados precisos e confiáveis.

Por que o Parameter Tuning é importante?

O Parameter Tuning é importante porque os parâmetros de um modelo de machine learning têm um impacto significativo em seu desempenho. A escolha dos parâmetros certos pode fazer a diferença entre um modelo que produz resultados precisos e um modelo que é inútil. Ao ajustar os parâmetros de um modelo, os cientistas de dados podem melhorar sua capacidade de fazer previsões precisas e identificar padrões nos dados.

Como funciona o Parameter Tuning?

O Parameter Tuning envolve a seleção dos melhores valores para os parâmetros de um modelo de machine learning. Isso geralmente é feito por meio de técnicas de otimização, como grid search, random search ou otimização bayesiana. O objetivo é encontrar a combinação ideal de parâmetros que maximize a precisão do modelo e minimize o erro de predição.

https://iatracker.com.br/glossario/o-que-e-parameter-tuning/#:~:text=Parameter%20Tuning%2C%20ou%20ajuste%20de,melhorar%20sua%20precis%C3%A3o%20e%20desempenho.

https://aws.amazon.com/what-is/hyperparameter-tuning/

#### Quais são as técnicas de ajuste de hiperparâmetros?

Existem diversos algoritmos de ajuste de hiperparâmetros, embora os tipos mais comumente usados sejam a otimização bayesiana, o grid search e o randomized search.

#### Otimização bayesiana

A otimização bayesiana é uma técnica baseada no teorema de Bayes, que descreve a probabilidade de ocorrência de um evento relacionado ao conhecimento atual. Quando isso é aplicado à otimização de hiperparâmetros, o algoritmo cria um modelo probabilístico a partir de um conjunto de hiperparâmetros que otimiza uma métrica específica. Ele usa análise de regressão para escolher iterativamente o melhor conjunto de hiperparâmetros.

#### Pesquisa em grade

Com a pesquisa em grade, você especifica uma lista de hiperparâmetros e uma métrica de desempenho, e o algoritmo trabalha com todas as combinações possíveis para determinar o melhor ajuste. A pesquisa em grade funciona bem, mas é relativamente tediosa e consome muitos recursos computacionais, especialmente com um grande número de hiperparâmetros.

#### Pesquisa aleatória

Embora baseada em princípios semelhantes aos da pesquisa em grade, a pesquisa aleatória seleciona grupos de hiperparâmetros aleatoriamente em cada iteração. Funciona bem quando um número relativamente pequeno de hiperparâmetros determina principalmente o resultado do modelo.

# Projeto 3: Classificação binária brest cancer com tuning dos parâmetros

## Etapa 1: Importação das bibliotecas

usar o recurso do skorch, pois o pytorch não possui o processo. 

In [31]:
# no linux o ambiente sera YouTube

# conda env list
# conda activate Youtube


In [1]:
#!pip install skorch

In [4]:
import pandas as pd
import numpy as np
import sklearn
import skorch
import torch
import torch.nn as nn
import torch.nn.functional as F # funções de ativações
from sklearn.model_selection import GridSearchCV
from skorch import NeuralNetBinaryClassifier

In [3]:
print('torch', torch.__version__, '\nskorch', skorch.__version__, '\nsklearn',sklearn.__version__)

torch 2.8.0+cu128 
skorch 1.2.0 
sklearn 1.4.2


## Etapa 2: Base de dados

In [5]:
# os mesmos experimentos, para ficar os mesmos quando reprocessarmos. 
np.random.seed(123)
torch.manual_seed(123)


In [9]:
# recarregar a base de dados
previsores = pd.read_csv('entradas_breast.csv')
classe = pd.read_csv('saidas_breast.csv')

print(previsores.tail(2)), print('*'*20), print(classe.tail(2))

      radius_mean   texture_mean   perimeter_mean   area_mean  \
567         20.60          29.33           140.10      1265.0   
568          7.76          24.54            47.92       181.0   

      smoothness_mean   compactness_mean   concavity_mean  \
567           0.11780          277.00000           0.3514   
568           0.05263            0.04362           0.0000   

     concave_points_mean   symmetry_mean   fractal_dimension_mean  ...  \
567                152.0          0.2397                  0.07016  ...   
568                  0.0          0.1587                  0.05884  ...   

      radius_worst   texture_worst   perimeter_worst   area_worst  \
567          25.74           39.42            184.60       1821.0   
568        9456.00           30.37             59.16        268.6   

      smoothness_worst   compactness_worst   concavity_worst  \
567          165.00000             0.86810            0.9387   
568            0.08996             0.06444            0.0000 

(None, None, None)

In [11]:
classe.shape

(569, 1)

In [12]:
# conversão para array e float32 para trabalharmos no sklearn
previsores = np.array(previsores, dtype = 'float32')
classe = np.array(classe, dtype = 'float32').squeeze(1)

In [13]:
previsores.shape

(569, 30)

In [14]:
classe.shape

(569,)

rate do pytorch com o sklearn

## Etapa 3: Classe para estrutura da rede neural


**\*\* ATUALIZAÇÃO JAN/2022 \*\*** : na versão atual do Skorch, os resultados da rede neural devem ser retornados sem ativação, ou seja, sem a camada sigmoide no final. Com isto, a função de custo deve ser `BCEWithLogitsLoss`.

In [23]:
# module para conseguir a compatibilidade
class classificador_torch(nn.Module):
  # init inicializar o programa, construtor da classe
  # self parâmetro padrão, e padrões adicionais,
  # parâmetros para o gridsearch buscar o melhor
  # passaremos a função de ativação => activation
  # passar a quantidade de neurônios => neurons
  # inicialização dos pesos => initializer   
  # ideia, number_of_layers, número de camadas, caso queremos colocar para escolher mais de 2 camadas. Para a estrutura de repetição
  def __init__(self, activation, neurons, initializer): # passado o parâmetro para escolher e ver os passados.
    super().__init__() # super classe por module
    # estrutura camada de entrada (30 neurônios), camada oculta (1°, 16 neurônios) e camada oculta (2°, 16 neurônios) e camada de saída (1 neurônio)
    # 30 -> 16 -> 16 -> 1
    # porém, agora não vamos colocar 16 neurônios, na camada oculta, 
    # e sim a variável neurons para escolher os melhores neurônios nas camadas ocultas.   
    # densidade de forma linear
    self.dense0 = nn.Linear(30, neurons)
    # escolha dos pesos
    initializer(self.dense0.weight)
    # no momento não passamos o valor ReLU e sim a ativação que será escolhida como melhor parâmetros.
    self.activation0 = activation
    # escolha no processo linear das quantidades de neurônios nas camadas ocultas primeira e segunda. 
    self.dense1 = nn.Linear(neurons, neurons)
    # pesos entre as camadas
    initializer(self.dense1.weight)
    # ativação 
    self.activation1 = activation
    self.dense2 = nn.Linear(neurons, 1)
    # cálculo dos pesos entre a camada oculta e a camada de saída.
    initializer(self.dense2.weight)
    # self.output = nn.Sigmoid() ** ATUALIZAÇÃO (ver detalhes no texto acima) **

  def forward(self, X):
    # X é a entrada.
    X = self.dense0(X)
    X = self.activation0(X)
    X = self.dense1(X)
    X = self.activation1(X)
    X = self.dense2(X)
    # X = self.output(X) ** ATUALIZAÇÃO (ver detalhes no texto acima) **
    return X

## Etapa 4: Skorch

In [24]:
classificador_sklearn = NeuralNetBinaryClassifier(module=classificador_torch, # modulo recebe o classificador (Cérebro do classificador da rede neural)
                                                  lr = 0.001, # learning rate, taxa de aprendizagem
                                                  optimizer__weight_decay = 0.0001,
                                                  train_split=False) # para parar a validação cruzada agora, pois faremos em outro local
                                                                     # a validação cruzada

#### optimizer__weight_decay=0.0001:

Este parâmetro se refere à regularização L2 (também conhecida como weight decay). Ela adiciona um termo de penalidade à função de perda que é proporcional ao quadrado dos pesos da rede neural.

Impacto: Ajuda a prevenir o overfitting, desencorajando pesos muito grandes na rede. Isso torna o modelo mais "simples" e mais propenso a generalizar para dados não vistos. 0.0001 é um valor pequeno, indicando uma regularização leve.



## Etapa 5: Tuning dos parâmetros

In [27]:
#configuração dos parâmetros, colocando na forma de dicionário:

params = {'batch_size': [10,30],
          'max_epochs': [50,100],
          'optimizer': [torch.optim.Adam, torch.optim.SGD], # adam seria uma melhoria
          'criterion': [torch.nn.BCEWithLogitsLoss], #, torch.nn.HingeEmbeddingLoss], # ** ATUALIZAÇÃO ** # cálcula o erro
          'module__activation': [F.relu, F.tanh],
          'module__neurons': [8, 16],
          'module__initializer': [torch.nn.init.uniform]} # _, torch.nn.init.normal_]} # inicializador dos pesos



### Análise dos Parâmetros

- 'batch_size': Define o número de amostras de treino que serão processadas antes que os pesos do modelo sejam atualizados. Valores de [10, 30] serão testados.

- 'max_epochs': É o número máximo de vezes que o modelo irá "ver" todo o conjunto de dados de treino durante o treinamento. A busca testará rodar o treinamento por 50 ou 100 épocas.

- 'optimizer': O otimizador é o algoritmo que ajusta os pesos do modelo para minimizar o erro. O Grid Search irá comparar o desempenho do modelo usando o otimizador Adam e o SGD (Gradiente Descendente Estocástico).

- 'criterion': Esta é a função de perda ou de custo. Ela mede o quão errado o modelo está. Seu código está configurado para usar a BCEWithLogitsLoss, que é uma função de perda comum para problemas de classificação binária.

- 'module__activation': Define a função de ativação que será usada nas camadas da rede neural. A busca irá testar o uso de ReLU e Tanh. O prefixo module__ é uma convenção para indicar que este parâmetro pertence ao módulo PyTorch que foi passado para o NeuralNetBinaryClassifier.

- 'module__neurons': Define o número de neurônios nas camadas ocultas da rede. A busca testará modelos com 8 ou 16 neurônios.

- 'module__initializer': Define como os pesos da rede neural serão inicializados. A busca irá testar a inicialização uniforme, que pode influenciar a velocidade e a estabilidade do treinamento. Mas, também pode testar a inicialização normal.



In [29]:
params # pode testar um a um, do que testar tudo de uma vez. 

{'batch_size': [10, 30],
 'max_epochs': [50, 100],
 'optimizer': [torch.optim.adam.Adam, torch.optim.sgd.SGD],
 'criterion': [torch.nn.modules.loss.BCEWithLogitsLoss],
 'module__activation': [<function torch.nn.functional.relu(input: torch.Tensor, inplace: bool = False) -> torch.Tensor>,
  <function torch.nn.functional.tanh(input)>],
 'module__neurons': [8, 16],
 'module__initializer': [<function torch.nn.init._make_deprecate.<locals>.deprecated_init(*args: _P.args, **kwargs: _P.kwargs) -> ~_R>]}

In [30]:
grid_search = GridSearchCV(estimator=classificador_sklearn, param_grid=params,
                           scoring = 'accuracy', cv = 2) # validação cruzada com 2 teste, o padrão ideal é 10. Treina 1 e testa 1 porção. 
# fit é para encaixar os dados, e cria as estruturas das redes neurais.
grid_search = grid_search.fit(previsores, classe)

/tmp/ipykernel_38858/1104656523.py:19: FutureWarning: `nn.init.uniform` is now deprecated in favor of `nn.init.uniform_`.
  initializer(self.dense0.weight)
/tmp/ipykernel_38858/1104656523.py:25: FutureWarning: `nn.init.uniform` is now deprecated in favor of `nn.init.uniform_`.
  initializer(self.dense1.weight)
/tmp/ipykernel_38858/1104656523.py:30: FutureWarning: `nn.init.uniform` is now deprecated in favor of `nn.init.uniform_`.
  initializer(self.dense2.weight)


  epoch    train_loss     dur
-------  ------------  ------
      1    27299.8736  0.0376
      2    24343.2034  0.0239
      3    21607.9790  0.0203
      4    19135.1196  0.0197
      5    16931.4971  0.0222
      6    14986.2345  0.0246
      7    13263.7452  0.0159
      8    11731.9724  0.0247
      9    10367.9490  0.0205
     10     9150.4826  0.0200
     11     8060.7256  0.0209
     12     7080.4180  0.0211
     13     6192.2901  0.0179
     14     5382.8819  0.0401
     15     4640.1386  0.0401
     16     3952.5591  0.0394
     17     3309.9036  0.0455
     18     2702.8657  0.0709
     19     2122.0785  0.0541
     20     1558.6171  0.0452
     21     1003.4027  0.0952
     22      478.4378  0.0610
     23      138.4220  0.0532
     24       90.7287  0.0455
     25       88.3918  0.0449
     26       77.7050  0.0475
     27       70.4366  0.0443
     28       64.3088  0.0453
     29       58.6979  0.0501
     30       54.5392  0.0459
     31       52.0841  0.0446
     32   

/tmp/ipykernel_38858/1104656523.py:19: FutureWarning: `nn.init.uniform` is now deprecated in favor of `nn.init.uniform_`.
  initializer(self.dense0.weight)
/tmp/ipykernel_38858/1104656523.py:25: FutureWarning: `nn.init.uniform` is now deprecated in favor of `nn.init.uniform_`.
  initializer(self.dense1.weight)
/tmp/ipykernel_38858/1104656523.py:30: FutureWarning: `nn.init.uniform` is now deprecated in favor of `nn.init.uniform_`.
  initializer(self.dense2.weight)


      7    14310.4431  0.0474
      8    12627.0749  0.0341
      9    11098.6747  0.0226
     10     9706.9926  0.0182
     11     8434.7943  0.0238
     12     7264.4389  0.0411
     13     6185.1322  0.0402
     14     5186.7830  0.0407
     15     4250.3310  0.0389
     16     3362.9720  0.0396
     17     2533.7989  0.0480
     18     1763.0047  0.0453
     19     1031.2695  0.0428
     20      462.3731  0.0428
     21      238.6089  0.0470
     22      214.4065  0.0575
     23      198.2918  0.0449
     24      181.4062  0.0469
     25      171.1930  0.0436
     26      161.6545  0.0420
     27      153.4939  0.0417
     28      146.7081  0.0449
     29      138.1020  0.0441
     30      128.5929  0.0403
     31      125.4274  0.0418
     32      118.2038  0.0394
     33      110.6343  0.0389
     34      101.7886  0.0383
     35       94.7599  0.0403
     36       85.2593  0.0416
     37       83.5713  0.0428
     38       77.4954  0.0393
     39       73.1303  0.0454
     40   

/tmp/ipykernel_38858/1104656523.py:19: FutureWarning: `nn.init.uniform` is now deprecated in favor of `nn.init.uniform_`.
  initializer(self.dense0.weight)
/tmp/ipykernel_38858/1104656523.py:25: FutureWarning: `nn.init.uniform` is now deprecated in favor of `nn.init.uniform_`.
  initializer(self.dense1.weight)
/tmp/ipykernel_38858/1104656523.py:30: FutureWarning: `nn.init.uniform` is now deprecated in favor of `nn.init.uniform_`.
  initializer(self.dense2.weight)


...

     18        3.5374  0.0106
     19        3.5136  0.0101
     20        3.4897  0.0101
     21        3.4659  0.0100
     22        3.4420  0.0101
     23        3.4181  0.0101
     24        3.3943  0.0101
     25        3.3704  0.0112
     26        3.3466  0.0102
     27        3.3227  0.0100
     28        3.2989  0.0104
     29        3.2750  0.0105
     30        3.2512  0.0100
     31        3.2273  0.0099
     32        3.2035  0.0102
     33        3.1796  0.0104
     34        3.1558  0.0103
     35        3.1319  0.0101
     36        3.1081  0.0101
     37        3.0842  0.0100
     38        3.0604  0.0102
     39        3.0366  0.0102
     40        3.0127  0.0101
     41        2.9889  0.0100
     42        2.9651  0.0102
     43        2.9412  0.0103
     44        2.9174  0.0112
     45        2.8936  0.0107
     46        2.8698  0.0103
     47        2.8460  0.0103
     48        2.8222  0.0104
     49        2.7984  0.0111
     50        2.7746  0.0101
     51   

/tmp/ipykernel_38858/1104656523.py:19: FutureWarning: `nn.init.uniform` is now deprecated in favor of `nn.init.uniform_`.
  initializer(self.dense0.weight)
/tmp/ipykernel_38858/1104656523.py:25: FutureWarning: `nn.init.uniform` is now deprecated in favor of `nn.init.uniform_`.
  initializer(self.dense1.weight)
/tmp/ipykernel_38858/1104656523.py:30: FutureWarning: `nn.init.uniform` is now deprecated in favor of `nn.init.uniform_`.
  initializer(self.dense2.weight)


     19        2.8218  0.0105
     20        2.7995  0.0100
     21        2.7772  0.0100
     22        2.7550  0.0102
     23        2.7327  0.0101
     24        2.7104  0.0100
     25        2.6882  0.0100
     26        2.6659  0.0101
     27        2.6437  0.0103
     28        2.6214  0.0101
     29        2.5992  0.0101
     30        2.5770  0.0100
     31        2.5548  0.0102
     32        2.5326  0.0107
     33        2.5104  0.0101
     34        2.4882  0.0100
     35        2.4660  0.0100
     36        2.4438  0.0103
     37        2.4216  0.0100
     38        2.3995  0.0101
     39        2.3774  0.0100
     40        2.3552  0.0100
     41        2.3331  0.0102
     42        2.3110  0.0101
     43        2.2889  0.0101
     44        2.2668  0.0103
     45        2.2448  0.0100
     46        2.2228  0.0101
     47        2.2007  0.0101
     48        2.1787  0.0104
     49        2.1568  0.0107
     50        2.1348  0.0104
     51        2.1129  0.0100
     52   

/tmp/ipykernel_38858/1104656523.py:19: FutureWarning: `nn.init.uniform` is now deprecated in favor of `nn.init.uniform_`.
  initializer(self.dense0.weight)
/tmp/ipykernel_38858/1104656523.py:25: FutureWarning: `nn.init.uniform` is now deprecated in favor of `nn.init.uniform_`.
  initializer(self.dense1.weight)
/tmp/ipykernel_38858/1104656523.py:30: FutureWarning: `nn.init.uniform` is now deprecated in favor of `nn.init.uniform_`.
  initializer(self.dense2.weight)


      8    40564.0383  0.0285
      9    36306.5334  0.0275
     10    32437.7208  0.0274
     11    28922.2092  0.0270
     12    25735.8302  0.0271
     13    22841.0490  0.0270
     14    20204.6947  0.0287
     15    17803.0712  0.0272
     16    15613.2209  0.0274
     17    13610.5589  0.0274
     18    11771.1566  0.0273
     19    10076.4239  0.0302
     20     8508.1051  0.0280
     21     7047.5899  0.0272
     22     5676.8516  0.0271
     23     4378.8237  0.0271
     24     3136.2170  0.0272
     25     1931.7844  0.0270
     26      750.2962  0.0270
     27      147.6397  0.0272
     28      107.6403  0.0272
     29       96.4332  0.0272
     30       92.9543  0.0273
     31       83.0662  0.0274
     32       76.6463  0.0273
     33       70.0437  0.0288
     34       63.3282  0.0298
     35       57.2976  0.0277
     36       51.7574  0.0275
     37       45.7398  0.0274
     38       40.1090  0.0273
     39       39.3633  0.0273
     40       39.2512  0.0274
     41   

In [ ]:
# busca em grades, 

In [32]:
melhores_parametros = grid_search.best_params_
melhor_precisao = grid_search.best_score_

In [33]:
melhores_parametros

{'batch_size': 30,
 'criterion': torch.nn.modules.loss.BCEWithLogitsLoss,
 'max_epochs': 100,
 'module__activation': <function torch.nn.functional.relu(input: torch.Tensor, inplace: bool = False) -> torch.Tensor>,
 'module__initializer': <function torch.nn.init._make_deprecate.<locals>.deprecated_init(*args: _P.args, **kwargs: _P.kwargs) -> ~_R>,
 'module__neurons': 16,
 'optimizer': torch.optim.adam.Adam}

In [34]:
melhor_precisao

0.7643563133185075

In [36]:
print(f'A melhor precisão dos parâmetros foi de {melhor_precisao*100:.2f}%')

A melhor precisão dos parâmetros foi de 76.44%


In [38]:
from pprint import PrettyPrinter

my_printer_params = PrettyPrinter()
#my_printer_params = PrettyPrinter(indent=4, width=100, depth=3)
my_printer_params.pprint(melhores_parametros)

{'batch_size': 30,
 'criterion': <class 'torch.nn.modules.loss.BCEWithLogitsLoss'>,
 'max_epochs': 100,
 'module__activation': <function relu at 0x76f9e8532700>,
 'module__initializer': <function _make_deprecate.<locals>.deprecated_init at 0x76f9e85c47c0>,
 'module__neurons': 16,
 'optimizer': <class 'torch.optim.adam.Adam'>}


In [39]:
my_printer_params = PrettyPrinter(indent=4, width=100, depth=3)
my_printer_params.pprint(melhores_parametros)

{   'batch_size': 30,
    'criterion': <class 'torch.nn.modules.loss.BCEWithLogitsLoss'>,
    'max_epochs': 100,
    'module__activation': <function relu at 0x76f9e8532700>,
    'module__initializer': <function _make_deprecate.<locals>.deprecated_init at 0x76f9e85c47c0>,
    'module__neurons': 16,
    'optimizer': <class 'torch.optim.adam.Adam'>}
